# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Two paper findings + my methodology questions
Research paper used: `docs/flyrank-seo-research-march-2026.pdf`.

**Finding 1 — The Anatomy of Growing Content.** The paper compared pages with rising versus falling impressions. Growing pages averaged 3,180 words and 184 days old, while declining pages averaged 2,311 words and 230 days old. The label came from the observed 30-day impression trend: pages were classified as growing or declining from the change between the last 30 days and the previous 30 days. This supports an observed association, not proof that adding words or refreshing a page will cause growth. My methodology question is whether this pattern remains when clients are held out, rather than allowing pages from the same client in both train and test.

**Finding 2 — The Content Performance Curve.** The paper reported that content performed best around 61–90 days and showed a decline around 271–365 days, with a higher 365+ result partly concentrated among older pages that had been refreshed. The outcome was the paper’s composite health score, paired with raw search-performance measures. My methodology question is whether the age pattern is stable across clients and content types, and whether the 365+ rebound reflects refresh selection or survivor bias rather than an age effect.

The paper says its headline findings come mainly from direct portfolio comparisons, while its ML appendix is exploratory. That is a useful standard for my project: the validation below can support out-of-sample ranking of an observed decline proxy, but it cannot support a causal claim that refreshing a page will increase traffic.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())

def make_features(frame, columns):
    numeric = [c for c in columns if c not in {"content_type", "main_intent"}]
    categorical = [c for c in columns if c in {"content_type", "main_intent"}]
    numeric_frame = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    categorical_frame = pd.get_dummies(frame[categorical].fillna("unknown").astype(str), prefix=categorical, dtype=float)
    return pd.concat([numeric_frame.reset_index(drop=True), categorical_frame.reset_index(drop=True)], axis=1)

ROOT = find_root()
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
df["label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
feature_columns = ["search_volume", "competition", "cpc", "content_type", "main_intent", "word_count", "char_count", "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "content_age_days", "age_tier_order", "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
feature_columns = [c for c in feature_columns if c in df.columns]
X = make_features(df, feature_columns)
y = df["label"]
print(f"Rows: {len(df):,} | decline base rate: {y.mean():.3f}")

Rows: 30,000 | decline base rate: 0.542


## 2. My model under an honest split (before/after)

The “before” result uses a stratified random row split. The “after” result holds out whole clients. The grouped result is more honest because pages from the same client can share hidden traffic patterns, content practices, and measurement conditions. I will report both so the gap itself is visible.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

all_indices = np.arange(len(df))
train_random, test_random = train_test_split(all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
clients = df["client_id"].fillna("unknown").astype(str)
rng = np.random.default_rng(RANDOM_STATE)
held_out_clients = set(rng.permutation(clients.drop_duplicates().to_numpy())[:6])
test_group_mask = clients.isin(held_out_clients).to_numpy()
train_group = all_indices[~test_group_mask]
test_group = all_indices[test_group_mask]

model_random = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)
model_group = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)
model_random.fit(X.iloc[train_random], y.iloc[train_random])
model_group.fit(X.iloc[train_group], y.iloc[train_group])
random_scores = model_random.predict_proba(X.iloc[test_random])[:, 1]
group_scores = model_group.predict_proba(X.iloc[test_group])[:, 1]

results = pd.DataFrame([
    {"split": "random_row_holdout", "test_rows": len(test_random), "precision_at_20": precision_at_k(random_scores, y.iloc[test_random], 20), "precision_at_50": precision_at_k(random_scores, y.iloc[test_random], 50), "average_precision": average_precision_score(y.iloc[test_random], random_scores), "roc_auc": roc_auc_score(y.iloc[test_random], random_scores), "base_rate": y.iloc[test_random].mean()},
    {"split": "client_holdout", "test_rows": len(test_group), "precision_at_20": precision_at_k(group_scores, y.iloc[test_group], 20), "precision_at_50": precision_at_k(group_scores, y.iloc[test_group], 50), "average_precision": average_precision_score(y.iloc[test_group], group_scores), "roc_auc": roc_auc_score(y.iloc[test_group], group_scores), "base_rate": y.iloc[test_group].mean()},
])
results.round(3)

,split,test_rows,precision_at_20,precision_at_50,average_precision,roc_auc,base_rate
0,random_row_holdout,6000,1.0,0.98,0.802,0.798,0.542
1,client_holdout,2325,0.9,0.92,0.801,0.893,0.391


## 3. Leakage audit

The final feature set must not contain the field that defines the label, its sibling fields, future-window outcomes, identifiers, or product decision scores. I will also run a deliberate “bad” test: adding `trend_direction` should make the score look suspiciously strong because the label is defined from that field. That feature is then removed and the honest result is retained.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
forbidden = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id", "clicks_last_30d", "sessions_last_30d"}
assert forbidden.isdisjoint(set(feature_columns))
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns

leaky_frame = df[["trend_direction"]].copy()
leaky_frame["leaky_score"] = leaky_frame["trend_direction"].eq("down").astype(float)
leaky_precision = precision_at_k(leaky_frame["leaky_score"], y, 100)
print("Forbidden feature check passed.")
print(f"Deliberate trend-field leakage Precision@100: {leaky_precision:.3f}")
print("This is expected to be near-perfect because trend_direction defines the label; it is not a valid model result.")
print("The retained model uses only pre-decision/content fields and evaluates on held-out clients.")

Forbidden feature check passed.
Deliberate trend-field leakage Precision@100: 1.000
This is expected to be near-perfect because trend_direction defines the label; it is not a valid model result.
The retained model uses only pre-decision/content fields and evaluates on held-out clients.


## 4. Claim rewrite

Unsafe claim: “The model predicts which pages will recover after a refresh.”

Safe claim: “On this anonymized starter dataset, under a client-held-out evaluation, the model ranked pages labeled as declining better than the transparent baseline at the tested top-K levels. This is directional decision support for review prioritization; it does not show that refreshing a page will cause traffic recovery.”

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
safe_claim = (
    "In this anonymized starter dataset, the random-forest model ranked observed decline labels "
    "on unseen clients with measured precision at the tested top-K levels. This supports directional "
    "decision support for review prioritization; it does not show that refreshing a page causes traffic recovery."
)
print(safe_claim)
print("Base rate used for context:", round(float(y.mean()), 3))
assert "causes" in safe_claim
assert "recovery" in safe_claim
assert results["split"].tolist() == ["random_row_holdout", "client_holdout"]
print("Claim audit passed.")

In this anonymized starter dataset, the random-forest model ranked observed decline labels on unseen clients with measured precision at the tested top-K levels. This supports directional decision support for review prioritization; it does not show that refreshing a page causes traffic recovery.
Base rate used for context: 0.542
Claim audit passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.